# PulseFit Leisure: generating a synthetic gym chain database

This notebook creates `pulsefit_gyms.db`, a SQLite database describing a fictional chain of six gyms in the
East of England. Every value is produced inside this notebook. Nothing is downloaded or copied from an
existing dataset.

The database covers the sites, the membership plans on sale, the members, the instructors and the classes
they are qualified to teach, the timetable of class sessions from January to September 2026, the bookings
members make for those sessions, and periodic body assessments carried out by staff.

**Approach.** Each table has its own generator function that returns a pandas DataFrame. The functions are
called in parent-to-child order so that every foreign key already exists when a child row is created. A
single NumPy `Generator` seeded with 3141 drives all randomness, which means the notebook rebuilds an
identical database each time it is run.

**Layout of the notebook**

| Part | Content |
|---|---|
| A | Libraries, seed and constants |
| B | Table definitions (DDL) |
| C | Lookup data written by hand |
| D | Generator functions, one per table |
| E | Building the tables |
| F | Deliberate data-quality problems |
| G | Writing to SQLite and checking the result |
| H | Example analysis queries |

In [1]:
try:
    import faker
except ImportError:
    %pip install -q faker

## A. Libraries, seed and constants

In [2]:
import os
import sqlite3
from datetime import date, datetime, timedelta

import numpy as np
import pandas as pd
from faker import Faker

SEED = 3141
gen = np.random.default_rng(SEED)
fk = Faker('en_GB')
Faker.seed(SEED)

DB_FILE = 'pulsefit_gyms.db'
TIMETABLE_START = date(2026, 1, 5)    # first Monday of 2026
TIMETABLE_WEEKS = 38                  # up to late September 2026
TODAY = date(2026, 9, 30)

## B. Table definitions

The schema is written out in full so the constraints can be read in one place. Points worth noting:

* `bookings` and `instructor_skills` both use **composite primary keys**, because each is a link between two
  other tables and a pair should only appear once.
* `class_sessions` has a **composite UNIQUE** constraint on `(instructor_id, starts_at)`, since an instructor
  cannot lead two classes at the same moment.
* `members.email` and the assessment table have no uniqueness rule on purpose. Part F inserts duplicate
  records there, and a unique index would reject them.
* Dates are stored as ISO text and validated with SQLite's `date()` and `datetime()` functions.

In [3]:
DDL = """
PRAGMA foreign_keys = ON;

CREATE TABLE sites (
    site_id         INTEGER PRIMARY KEY,
    site_name       TEXT    NOT NULL UNIQUE,
    town            TEXT    NOT NULL,
    county          TEXT    NOT NULL CHECK (county IN ('Hertfordshire','Cambridgeshire','Essex','Bedfordshire')),
    opened_year     INTEGER NOT NULL CHECK (opened_year BETWEEN 1985 AND 2026),
    floor_area_m2   REAL    NOT NULL CHECK (floor_area_m2 BETWEEN 300 AND 5000),
    has_pool        INTEGER NOT NULL DEFAULT 0 CHECK (has_pool IN (0,1))
);

CREATE TABLE membership_plans (
    plan_code          TEXT    PRIMARY KEY CHECK (plan_code GLOB 'PL[0-9][0-9]'),
    plan_name          TEXT    NOT NULL UNIQUE,
    tier               TEXT    NOT NULL CHECK (tier IN ('Off-peak','Standard','Premium','Elite')),
    monthly_fee_gbp    REAL    NOT NULL CHECK (monthly_fee_gbp > 0),
    min_term_months    INTEGER NOT NULL CHECK (min_term_months IN (1,3,12)),
    multi_site_access  INTEGER NOT NULL DEFAULT 0 CHECK (multi_site_access IN (0,1))
);

CREATE TABLE members (
    member_id       INTEGER PRIMARY KEY,
    first_name      TEXT    NOT NULL CHECK (length(first_name) > 0 AND length(first_name) <= 38),
    last_name       TEXT    NOT NULL CHECK (length(last_name) > 0 AND length(last_name) <= 38),
    email           TEXT    NOT NULL CHECK (instr(email, '@') > 1 AND instr(email, ' ') = 0),
    mobile          TEXT    CHECK (mobile IS NULL OR (instr(mobile, ' ') = 6 AND length(mobile) = 12 AND mobile GLOB '07[0-9]* [0-9]*')),
    birth_year      INTEGER NOT NULL CHECK (birth_year BETWEEN 1940 AND 2010),
    sex             TEXT    NOT NULL CHECK (sex IN ('Female','Male','Prefer not to say')),
    home_site_id    INTEGER NOT NULL REFERENCES sites(site_id) ON DELETE RESTRICT,
    plan_code       TEXT    NOT NULL REFERENCES membership_plans(plan_code) ON UPDATE CASCADE ON DELETE RESTRICT,
    joined_on       TEXT    NOT NULL CHECK (strftime('%Y-%m-%d', joined_on) = joined_on),
    fitness_goal    TEXT    CHECK (fitness_goal IS NULL OR fitness_goal IN
                        ('Lose weight','Build strength','Improve fitness','Rehabilitation','Sport performance','Social')),
    status          TEXT    NOT NULL DEFAULT 'Active' CHECK (status IN ('Active','Frozen','Cancelled'))
);

CREATE TABLE instructors (
    instructor_id   INTEGER PRIMARY KEY,
    site_id         INTEGER NOT NULL REFERENCES sites(site_id) ON DELETE RESTRICT,
    first_name      TEXT    NOT NULL,
    last_name       TEXT    NOT NULL,
    reps_level      TEXT    NOT NULL CHECK (reps_level IN ('Level 2','Level 3','Level 4')),
    qualified_year  INTEGER NOT NULL CHECK (qualified_year BETWEEN 1995 AND 2026),
    hourly_rate_gbp REAL    NOT NULL CHECK (hourly_rate_gbp BETWEEN 11.44 AND 60)
);

CREATE TABLE class_types (
    class_code     TEXT    PRIMARY KEY CHECK (class_code GLOB '[A-Z][A-Z][A-Z]'),
    class_name     TEXT    NOT NULL UNIQUE,
    category       TEXT    NOT NULL CHECK (category IN ('Cardio','Strength','Mind and body','Aqua','Cycle')),
    intensity      INTEGER NOT NULL CHECK (intensity BETWEEN 1 AND 5),
    duration_min   INTEGER NOT NULL CHECK (duration_min IN (30,45,60,75)),
    needs_pool     INTEGER NOT NULL DEFAULT 0 CHECK (needs_pool IN (0,1))
);

CREATE TABLE instructor_skills (
    instructor_id  INTEGER NOT NULL REFERENCES instructors(instructor_id) ON DELETE CASCADE,
    class_code     TEXT    NOT NULL REFERENCES class_types(class_code) ON DELETE CASCADE,
    certified_on   TEXT    NOT NULL CHECK (julianday(certified_on) IS NOT NULL AND length(certified_on) = 10),
    PRIMARY KEY (instructor_id, class_code)
);

CREATE TABLE class_sessions (
    session_id     INTEGER PRIMARY KEY,
    class_code     TEXT    NOT NULL REFERENCES class_types(class_code) ON DELETE RESTRICT,
    instructor_id  INTEGER NOT NULL REFERENCES instructors(instructor_id) ON DELETE RESTRICT,
    site_id        INTEGER NOT NULL REFERENCES sites(site_id) ON DELETE RESTRICT,
    starts_at      TEXT    NOT NULL CHECK (strftime('%Y-%m-%d %H:%M:%S', starts_at) = starts_at),
    capacity       INTEGER NOT NULL CHECK (capacity BETWEEN 6 AND 40),
    studio_temp_c  REAL    CHECK (studio_temp_c IS NULL OR studio_temp_c BETWEEN 14 AND 42),
    UNIQUE (instructor_id, starts_at)
);

CREATE TABLE bookings (
    session_id   INTEGER NOT NULL REFERENCES class_sessions(session_id) ON DELETE CASCADE,
    member_id    INTEGER NOT NULL REFERENCES members(member_id) ON DELETE CASCADE,
    booked_at    TEXT    NOT NULL CHECK (julianday(booked_at) IS NOT NULL AND length(booked_at) = 19),
    outcome      TEXT    NOT NULL CHECK (outcome IN ('Attended','No-show','Late cancel','Cancelled')),
    PRIMARY KEY (session_id, member_id)
);

CREATE TABLE body_assessments (
    assessment_id    INTEGER PRIMARY KEY,
    member_id        INTEGER NOT NULL REFERENCES members(member_id) ON DELETE CASCADE,
    assessed_on      TEXT    NOT NULL CHECK (date(assessed_on) = assessed_on),
    weight_kg        REAL    NOT NULL CHECK (weight_kg BETWEEN 35 AND 200),
    body_fat_pct     REAL    CHECK (body_fat_pct IS NULL OR body_fat_pct BETWEEN 4 AND 60),
    resting_hr_bpm   INTEGER CHECK (resting_hr_bpm IS NULL OR resting_hr_bpm BETWEEN 35 AND 120),
    self_rated_fitness INTEGER NOT NULL CHECK (self_rated_fitness BETWEEN 1 AND 5)
);

CREATE INDEX ix_bookings_member ON bookings(member_id);
CREATE INDEX ix_sessions_site_time ON class_sessions(site_id, starts_at);
"""

## C. Lookup data written by hand

These lists set the character of the chain: where the gyms are, what the plans cost and which classes run.
Plan prices are close to mid-market UK gym prices in 2026. The `HEIGHT` and `BMI` parameters are used later
to give each member a plausible body size, so that weight and body fat agree with each other.

In [4]:
SITE_ROWS = [
    ('PulseFit Hatfield', 'Hatfield', 'Hertfordshire', 2011, 1450, 1),
    ('PulseFit Stevenage Leisure Park', 'Stevenage', 'Hertfordshire', 2004, 2100, 1),
    ('PulseFit Cambridge Leisure', 'Cambridge', 'Cambridgeshire', 2016, 1800, 0),
    ('PulseFit Chelmsford Riverside', 'Chelmsford', 'Essex', 2019, 1250, 0),
    ('PulseFit Bedford Retail Park', 'Bedford', 'Bedfordshire', 2008, 980, 0),
    ('PulseFit St Albans', 'St Albans', 'Hertfordshire', 2022, 760, 0),
]
PLAN_ROWS = [  # code, name, tier, fee, term, multi-site, share of members
    ('PL01', 'Off-peak Flex', 'Off-peak', 22.99, 1, 0, .18),
    ('PL02', 'Standard Flex', 'Standard', 31.99, 1, 0, .27),
    ('PL03', 'Standard 12 Month', 'Standard', 27.99, 12, 0, .23),
    ('PL04', 'Premium Anywhere', 'Premium', 42.99, 3, 1, .20),
    ('PL05', 'Elite Coaching', 'Elite', 74.99, 3, 1, .05),
    ('PL06', 'Student Saver', 'Off-peak', 19.99, 3, 0, .07),
]
CLASS_ROWS = [  # code, name, category, intensity, minutes, pool
    ('HIT', 'HIIT Blast', 'Cardio', 5, 30, 0), ('CIR', 'Circuits', 'Cardio', 4, 45, 0),
    ('STP', 'Step', 'Cardio', 3, 45, 0), ('BPM', 'Body Pump', 'Strength', 4, 60, 0),
    ('KBL', 'Kettlebells', 'Strength', 4, 45, 0), ('COR', 'Core Conditioning', 'Strength', 3, 30, 0),
    ('YOG', 'Vinyasa Yoga', 'Mind and body', 2, 60, 0), ('HYG', 'Hot Yoga', 'Mind and body', 3, 60, 0),
    ('PIL', 'Pilates', 'Mind and body', 2, 45, 0), ('STR', 'Stretch and Release', 'Mind and body', 1, 30, 0),
    ('AQA', 'Aqua Aerobics', 'Aqua', 2, 45, 1), ('SWM', 'Swim Fit', 'Aqua', 3, 45, 1),
    ('SPN', 'Spin', 'Cycle', 4, 45, 0), ('RID', 'Rhythm Ride', 'Cycle', 3, 45, 0),
]
GOALS = ['Lose weight', 'Build strength', 'Improve fitness', 'Rehabilitation', 'Sport performance', 'Social']
GOAL_P = [.31, .22, .29, .05, .07, .06]
HEIGHT = {'Female': (164, 6.5), 'Male': (177, 7.0), 'Prefer not to say': (170, 8.5)}
SLOT_HOURS = [6.5, 7.0, 9.5, 10.0, 12.25, 17.5, 18.0, 18.75, 19.5, 20.0]
SLOT_WEIGHT = np.array([.9, 1.0, .6, .6, .5, 1.2, 1.4, 1.3, 1.1, .7])

## D. Generator functions

One function per table. Each takes what it needs from earlier tables and returns a DataFrame with the same
column names as the SQL table. Correlations that make the data look real are built in here:

* emails are made from the member's own name;
* the plan mix differs by age, with students most often on the Student Saver plan;
* body weight comes from height and a body mass index drawn per member, and body fat follows from BMI, sex
  and age (a simplified Deurenberg relationship), so the three measurements agree;
* sessions run at busier times in the evening and fewer on Sundays, studio temperature depends on the class
  (hot yoga is held at about 38 C);
* off-peak members can only book sessions that start before 16:00 on weekdays or at the weekend;
* attendance is more likely for higher plan tiers.

In [5]:
def make_sites():
    df = pd.DataFrame(SITE_ROWS, columns=['site_name', 'town', 'county', 'opened_year', 'floor_area_m2', 'has_pool'])
    df.insert(0, 'site_id', np.arange(1, len(df) + 1))
    return df


def make_plans():
    df = pd.DataFrame(PLAN_ROWS, columns=['plan_code', 'plan_name', 'tier', 'monthly_fee_gbp',
                                          'min_term_months', 'multi_site_access', 'share'])
    return df


def clean_token(text):
    return ''.join(ch for ch in text.lower() if ch.isalpha())


def make_members(sites, plans, n=2150):
    size_weight = sites.floor_area_m2 / sites.floor_area_m2.sum()
    rows = []
    for i in range(n):
        sex = gen.choice(['Female', 'Male', 'Prefer not to say'], p=[.51, .47, .02])
        first = fk.first_name_female() if sex == 'Female' else fk.first_name_male() if sex == 'Male' else fk.first_name()
        last = fk.last_name()
        age = int(np.clip(gen.gamma(5.5, 6.2), 16, 84))
        plan_p = plans.share.to_numpy().copy()
        if age <= 24:
            plan_p[5] *= 4.0
        else:
            plan_p[5] = 0.0
        if age >= 60:
            plan_p[0] *= 2.0
        plan_p /= plan_p.sum()
        style = gen.integers(0, 3)
        local = [f'{clean_token(first)}.{clean_token(last)}', f'{clean_token(first)[0]}{clean_token(last)}{gen.integers(1, 99)}',
                 f'{clean_token(last)}.{clean_token(first)}'][style]
        domain = gen.choice(['gmail.com', 'outlook.com', 'hotmail.co.uk', 'icloud.com', 'yahoo.co.uk', 'btinternet.com'],
                            p=[.41, .17, .15, .13, .09, .05])
        joined = TODAY - timedelta(days=int(min(gen.exponential(520), 4000)))
        rows.append({
            'first_name': first, 'last_name': last, 'email': f'{local}@{domain}',
            'mobile': f'07{gen.integers(300, 999)} {gen.integers(100000, 999999)}',
            'birth_year': 2026 - age, 'sex': sex,
            'home_site_id': int(gen.choice(sites.site_id, p=size_weight)),
            'plan_code': gen.choice(plans.plan_code, p=plan_p),
            'joined_on': joined.isoformat(),
            'fitness_goal': gen.choice(GOALS, p=GOAL_P),
            'status': gen.choice(['Active', 'Frozen', 'Cancelled'], p=[.86, .04, .10]),
        })
    df = pd.DataFrame(rows)
    # two different people can share a common name, birth year and site; keep only the first so that the
    # duplicate check in part G counts only the duplicates added on purpose in part F
    key = df.first_name.str.lower() + '|' + df.last_name.str.lower() + '|' + df.birth_year.astype(str) + '|' + df.home_site_id.astype(str)
    df = df[~key.duplicated()].reset_index(drop=True)
    df.insert(0, 'member_id', np.arange(1, len(df) + 1))
    return df


def make_instructors(sites):
    rows = []
    for s in sites.itertuples():
        for _ in range(int(round(s.floor_area_m2 / 250)) + 2):
            level = gen.choice(['Level 2', 'Level 3', 'Level 4'], p=[.35, .5, .15])
            years = {'Level 2': (2018, 2025), 'Level 3': (2008, 2023), 'Level 4': (1998, 2016)}[level]
            rate = {'Level 2': 14.5, 'Level 3': 19.0, 'Level 4': 27.0}[level] + gen.normal(0, 1.8)
            rows.append({'site_id': s.site_id, 'first_name': fk.first_name(), 'last_name': fk.last_name(),
                         'reps_level': level, 'qualified_year': int(gen.integers(years[0], years[1] + 1)),
                         'hourly_rate_gbp': round(float(max(rate, 12.21)), 2)})
    df = pd.DataFrame(rows)
    df.insert(0, 'instructor_id', np.arange(1, len(df) + 1))
    return df


def make_class_types():
    return pd.DataFrame(CLASS_ROWS, columns=['class_code', 'class_name', 'category', 'intensity', 'duration_min', 'needs_pool'])


def make_skills(instructors, classes, sites):
    pool_sites = set(sites.loc[sites.has_pool == 1, 'site_id'])
    rows = []
    for ins in instructors.itertuples():
        options = classes if ins.site_id in pool_sites else classes[classes.needs_pool == 0]
        k = int(gen.integers(2, 6))
        for code in gen.choice(options.class_code, size=min(k, len(options)), replace=False):
            cert = date(int(gen.integers(max(ins.qualified_year, 2010), 2026)), int(gen.integers(1, 13)), int(gen.integers(1, 28)))
            rows.append({'instructor_id': ins.instructor_id, 'class_code': code, 'certified_on': min(cert, date(2025, 12, 20)).isoformat()})
    return pd.DataFrame(rows)


def make_sessions(sites, skills, classes, instructors):
    temp_mean = {'Cardio': 19.5, 'Strength': 19.0, 'Mind and body': 22.0, 'Aqua': 29.5, 'Cycle': 18.5}
    cat = classes.set_index('class_code').category
    by_site = skills.merge(instructors[['instructor_id', 'site_id']], on='instructor_id').groupby('site_id')
    rows, busy = [], set()
    for s in sites.itertuples():
        pairs = by_site.get_group(s.site_id)[['instructor_id', 'class_code']].to_numpy()
        per_week = int(s.floor_area_m2 / 70)
        for week in range(TIMETABLE_WEEKS):
            for _ in range(per_week):
                day = TIMETABLE_START + timedelta(days=7 * week + int(gen.choice(7, p=[.16, .16, .16, .16, .13, .14, .09])))
                hour = float(gen.choice(SLOT_HOURS, p=SLOT_WEIGHT / SLOT_WEIGHT.sum()))
                start = datetime(day.year, day.month, day.day) + timedelta(hours=hour)
                ins_id, code = pairs[gen.integers(0, len(pairs))]
                if (ins_id, start) in busy:
                    continue
                busy.add((ins_id, start))
                if code == 'HYG':
                    temp = gen.normal(38.0, .8)
                else:
                    temp = gen.normal(temp_mean[cat[code]], 1.1)
                rows.append({'class_code': code, 'instructor_id': int(ins_id), 'site_id': s.site_id,
                             'starts_at': start.strftime('%Y-%m-%d %H:%M:%S'),
                             'capacity': int(gen.choice([12, 16, 20, 24, 30], p=[.15, .25, .3, .2, .1])),
                             'studio_temp_c': round(float(temp), 1)})
    df = pd.DataFrame(rows).sort_values('starts_at', kind='stable').reset_index(drop=True)
    df.insert(0, 'session_id', df.index + 1)
    return df


def make_bookings(sessions, members, plans):
    tier = members.merge(plans[['plan_code', 'tier', 'multi_site_access']], on='plan_code')
    tier = tier[tier.status != 'Cancelled'].copy()
    # how keen each member is: most come once or twice a week, a few come almost daily
    tier['keenness'] = gen.gamma(1.8, 1.0, size=len(tier))
    show_rate = {'Off-peak': .74, 'Standard': .79, 'Premium': .84, 'Elite': .9}
    start = pd.to_datetime(sessions.starts_at)
    offpeak_ok = (start.dt.hour < 16) | (start.dt.day_of_week >= 5)
    rows = []
    for s, t_ok, st in zip(sessions.itertuples(), offpeak_ok, start):
        eligible = tier[((tier.home_site_id == s.site_id) | (tier.multi_site_access == 1)) & (tier.joined_on < s.starts_at[:10])]
        if not t_ok:
            eligible = eligible[eligible.tier != 'Off-peak']
        demand = int(np.clip(gen.normal(s.capacity * .62, s.capacity * .2), 0, s.capacity))
        demand = min(demand, len(eligible))
        if demand == 0:
            continue
        weight = eligible.keenness * np.where(eligible.home_site_id == s.site_id, 1.0, 0.12)
        picks = eligible.sample(n=demand, weights=weight, random_state=int(gen.integers(0, 1_000_000)))
        for m in picks.itertuples():
            lead_h = float(gen.gamma(1.6, 30))
            booked = st - timedelta(hours=min(lead_h, 160))
            r = gen.random()
            p_show = show_rate[m.tier]
            outcome = 'Attended' if r < p_show else 'No-show' if r < p_show + .09 else 'Late cancel' if r < p_show + .14 else 'Cancelled'
            rows.append({'session_id': s.session_id, 'member_id': m.member_id,
                         'booked_at': booked.strftime('%Y-%m-%d %H:%M:%S'), 'outcome': outcome})
    return pd.DataFrame(rows)


def make_assessments(members, bookings):
    attended = bookings[bookings.outcome == 'Attended'].groupby('member_id').size()
    rows = []
    for m in members.itertuples():
        visits = int(attended.get(m.member_id, 0))
        n = int(gen.poisson(1.2 + visits / 25))
        if n == 0:
            continue
        mu, sd = HEIGHT[m.sex]
        height_m = gen.normal(mu, sd) / 100
        bmi = float(np.clip(gen.lognormal(np.log(26.0), .15), 17, 45))
        age = 2026 - m.birth_year
        joined = date.fromisoformat(m.joined_on)
        days = sorted(set(int(x) for x in gen.integers(0, max((TODAY - joined).days, 2), size=n)))
        trend = -0.004 if m.fitness_goal == 'Lose weight' else 0.0
        for k, d in enumerate(days):
            b = bmi * (1 + trend * k * 3) + gen.normal(0, .25)
            male = 1 if m.sex == 'Male' else 0
            fat = 1.2 * b + 0.23 * age - 10.8 * male - 5.4 + gen.normal(0, 2.0)
            hr = 74 - min(visits, 120) * .08 + (b - 25) * .4 + gen.normal(0, 6)
            rows.append({'member_id': m.member_id, 'assessed_on': (joined + timedelta(days=d)).isoformat(),
                         'weight_kg': round(float(max(b * height_m ** 2, 38.0)), 1),
                         'body_fat_pct': round(float(np.clip(fat, 5, 58)), 1),
                         'resting_hr_bpm': int(np.clip(round(hr), 40, 115)),
                         'self_rated_fitness': int(np.clip(round(3 + (min(visits, 80) / 40) - (b - 26) / 6 + gen.normal(0, .7)), 1, 5))})
    df = pd.DataFrame(rows)
    df.insert(0, 'assessment_id', np.arange(1, len(df) + 1))
    return df

## E. Building the tables

The generators are called in dependency order. `instructors` is built before the sessions function runs
because the timetable can only use instructors who hold the right skill.

In [6]:
sites = make_sites()
plans = make_plans()
members = make_members(sites, plans)
instructors = make_instructors(sites)
classes = make_class_types()
skills = make_skills(instructors, classes, sites)
sessions = make_sessions(sites, skills, classes, instructors)
bookings = make_bookings(sessions, members, plans)
assessments = make_assessments(members, bookings)

summary = pd.Series({'sites': len(sites), 'membership_plans': len(plans), 'members': len(members),
                     'instructors': len(instructors), 'class_types': len(classes), 'instructor_skills': len(skills),
                     'class_sessions': len(sessions), 'bookings': len(bookings), 'body_assessments': len(assessments)})
summary

sites                    6
membership_plans         6
members               2149
instructors             45
class_types             14
instructor_skills      160
class_sessions        4312
bookings             49887
body_assessments      4305
dtype: int64

In [7]:
members.groupby('plan_code').size().rename('members').to_frame().join(
    plans.set_index('plan_code')[['tier', 'monthly_fee_gbp']])

,members,tier,monthly_fee_gbp
plan_code,,,
PL01,399,Off-peak,22.99
PL02,589,Standard,31.99
PL03,484,Standard,27.99
PL04,420,Premium,42.99
PL05,105,Elite,74.99
PL06,152,Off-peak,19.99


## F. Deliberate data-quality problems

A real gym system would contain gaps and repeated records, so both are added here on purpose and at
realistic rates. Each change is logged in `quality_log` and printed below.

| Problem | Where | Rate | Typical real cause |
|---|---|---|---|
| Missing | members.mobile | 3.5% | member did not give a number at sign-up |
| Missing | members.fitness_goal | 6% | optional question skipped on the joining form |
| Missing | body_assessments.body_fat_pct | 4% | bioimpedance scale not available that day |
| Missing | body_assessments.resting_hr_bpm | 2% | reading not taken |
| Missing | class_sessions.studio_temp_c | 2.5% | studio sensor offline |
| Duplicate | members | 1.2% | person rejoined online after an old account lapsed, name typed with different case or spacing |
| Duplicate | body_assessments | 1% | the same assessment entered twice by staff |

In [8]:
quality_log = {}

def drop_values(frame, column, share, label=None):
    """Set a fixed share of a column to missing and record how many rows changed."""
    target = frame.index[frame[column].notna()]
    hit = gen.choice(target, size=int(round(len(target) * share)), replace=False)
    frame.loc[hit, column] = None
    quality_log[label or f'missing {column}'] = len(hit)

drop_values(members, 'mobile', .035)
drop_values(members, 'fitness_goal', .06)
drop_values(assessments, 'body_fat_pct', .04)
drop_values(assessments, 'resting_hr_bpm', .02)
drop_values(sessions, 'studio_temp_c', .025)

# duplicate members: a second account for the same person, created later, with a formatting change
originals = members.sample(frac=.012, random_state=SEED)
copies = originals.copy()
variant = np.arange(len(copies)) % 3
copies.loc[variant == 0, 'first_name'] = copies.loc[variant == 0, 'first_name'].str.upper()
copies.loc[variant == 1, 'last_name'] = copies.loc[variant == 1, 'last_name'] + ' '
copies.loc[variant == 2, 'email'] = copies.loc[variant == 2, 'email'].str.capitalize()
copies['joined_on'] = [min(date.fromisoformat(j) + timedelta(days=int(gen.integers(40, 400))), TODAY).isoformat()
                       for j in copies.joined_on]
copies['member_id'] = np.arange(members.member_id.max() + 1, members.member_id.max() + 1 + len(copies))
members = pd.concat([members, copies], ignore_index=True)
quality_log['duplicate members'] = len(copies)

# duplicate assessments: identical rows saved twice under a new id
twice = assessments.sample(frac=.01, random_state=SEED).copy()
twice['assessment_id'] = np.arange(assessments.assessment_id.max() + 1, assessments.assessment_id.max() + 1 + len(twice))
assessments = pd.concat([assessments, twice], ignore_index=True)
quality_log['duplicate assessments'] = len(twice)

pd.Series(quality_log, name='rows affected')

missing mobile             75
missing fitness_goal      129
missing body_fat_pct      172
missing resting_hr_bpm     86
missing studio_temp_c     108
duplicate members          26
duplicate assessments      43
Name: rows affected, dtype: int64

## G. Writing to SQLite and checking the result

Foreign keys are switched on before any insert, so a child row that points at a missing parent would stop
the load. The helper column `share` on the plans table is only used for sampling and is dropped here.

In [9]:
if os.path.exists(DB_FILE):
    os.remove(DB_FILE)
conn = sqlite3.connect(DB_FILE)
conn.executescript(DDL)
conn.execute('PRAGMA foreign_keys = ON')

def insert_frame(frame, table):
    frame = frame.astype(object).where(frame.notna(), None)
    placeholders = ', '.join('?' for _ in frame.columns)
    conn.executemany(f'INSERT INTO {table} ({", ".join(frame.columns)}) VALUES ({placeholders})',
                     frame.itertuples(index=False, name=None))

insert_frame(sites, 'sites')
insert_frame(plans.drop(columns='share'), 'membership_plans')
insert_frame(members, 'members')
insert_frame(instructors, 'instructors')
insert_frame(classes, 'class_types')
insert_frame(skills, 'instructor_skills')
insert_frame(sessions, 'class_sessions')
insert_frame(bookings, 'bookings')
insert_frame(assessments, 'body_assessments')
conn.commit()

def sql(statement):
    return pd.read_sql_query(statement, conn)
tables = [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY rowid")]
pd.DataFrame({'table': tables,
              'columns': [len(conn.execute(f'PRAGMA table_info({t})').fetchall()) for t in tables],
              'rows': [conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0] for t in tables]})

,table,columns,rows
0,sites,7,6
1,membership_plans,6,6
2,members,12,2175
3,instructors,7,45
4,class_types,6,14
5,instructor_skills,3,160
6,class_sessions,7,4312
7,bookings,4,49887
8,body_assessments,7,4348


In [10]:
print('integrity_check :', conn.execute('PRAGMA integrity_check').fetchone()[0])
print('foreign_key_check rows :', len(conn.execute('PRAGMA foreign_key_check').fetchall()))

integrity_check : ok
foreign_key_check rows : 0


The next two queries find the injected problems again with SQL only. Duplicate members are matched on a
normalised name plus date of birth year and home site; duplicate assessments are rows that agree on every
measured value.

In [11]:
sql("""
SELECT
  (SELECT COUNT(*) - COUNT(DISTINCT lower(trim(first_name)) || '|' || lower(trim(last_name)) || '|' || birth_year || '|' || home_site_id)
     FROM members) AS duplicate_members,
  (SELECT COUNT(*) FROM (SELECT 1 FROM body_assessments
       GROUP BY member_id, assessed_on, weight_kg, IFNULL(body_fat_pct, -1), IFNULL(resting_hr_bpm, -1), self_rated_fitness
       HAVING COUNT(*) > 1)) AS duplicate_assessments,
  (SELECT SUM(mobile IS NULL) FROM members) AS missing_mobile,
  (SELECT SUM(body_fat_pct IS NULL) FROM body_assessments) AS missing_body_fat,
  (SELECT SUM(studio_temp_c IS NULL) FROM class_sessions) AS missing_studio_temp
""")

,duplicate_members,duplicate_assessments,missing_mobile,missing_body_fat,missing_studio_temp
0,26,43,78,172,108


## H. Example analysis queries

In [12]:
sql("""
SELECT p.tier, COUNT(*) AS bookings,
       ROUND(100.0 * SUM(b.outcome = 'Attended') / COUNT(*), 1) AS attended_pct
FROM bookings b JOIN members m USING (member_id) JOIN membership_plans p USING (plan_code)
GROUP BY p.tier ORDER BY attended_pct DESC
""")

,tier,bookings,attended_pct
0,Elite,3032,90.2
1,Premium,15290,84.1
2,Standard,25043,79.6
3,Off-peak,6522,75.1


In [13]:
sql("""
SELECT ct.category, COUNT(*) AS sessions, ROUND(AVG(cs.studio_temp_c), 1) AS mean_temp_c,
       ROUND(AVG(fill.booked * 1.0 / cs.capacity) * 100, 1) AS mean_fill_pct
FROM class_sessions cs
JOIN class_types ct USING (class_code)
JOIN (SELECT session_id, COUNT(*) AS booked FROM bookings GROUP BY session_id) fill USING (session_id)
GROUP BY ct.category ORDER BY sessions DESC
""")

,category,sessions,mean_temp_c,mean_fill_pct
0,Mind and body,1346,26.1,58.7
1,Strength,1049,19.0,59.7
2,Cardio,827,19.5,59.6
3,Cycle,800,18.5,59.1
4,Aqua,284,29.6,60.5


In [14]:
sql("""
SELECT CASE WHEN 2026 - m.birth_year < 30 THEN 'under 30' WHEN 2026 - m.birth_year < 50 THEN '30 to 49' ELSE '50 plus' END AS age_band,
       m.sex, ROUND(AVG(a.weight_kg), 1) AS mean_weight_kg, ROUND(AVG(a.body_fat_pct), 1) AS mean_body_fat
FROM body_assessments a JOIN members m USING (member_id)
WHERE m.sex <> 'Prefer not to say'
GROUP BY age_band, m.sex ORDER BY age_band, m.sex
""")

,age_band,sex,mean_weight_kg,mean_body_fat
0,30 to 49,Female,71.3,35.1
1,30 to 49,Male,81.7,23.9
2,50 plus,Female,70.1,39.7
3,50 plus,Male,81.2,28.9
4,under 30,Female,70.5,31.2
5,under 30,Male,82.7,20.3


In [15]:
conn.close()
print(f'{DB_FILE} written, {os.path.getsize(DB_FILE) / 1e6:.1f} MB')

pulsefit_gyms.db written, 4.4 MB
